# Part C — Text Classification with RNNs / LSTMs

Six recurrent architectures applied to AG News (and IMDB for C.7):

| Variant | RNN cell | Layers | Bidirectional |
|---|---|---|---|
| 1RNN     | vanilla RNN | 1 | no  |
| 1Bi-RNN  | vanilla RNN | 1 | yes |
| 2Bi-RNN  | vanilla RNN | 2 | yes |
| 1LSTM    | LSTM        | 1 | no  |
| 1Bi-LSTM | LSTM        | 1 | yes |
| 2Bi-LSTM | LSTM        | 2 | yes |

All models share these hyper-parameters (per the assignment):
`epochs=15`, `batch_size=1024`, `embedding_dim=100`, `hidden_dim=64`, `lr=1e-3`, vocab `min_freq=10`.

**This notebook does not retrain.** All experiments were run separately by the per-question driver scripts (`c1_baseline.py` … `c7_imdb.py`); their JSON results are loaded here and visualized.
Re-running everything from this notebook would take ~1 hour on Apple-MPS / ~10 min on a Colab T4.

## Setup

Add `part_c_rnn_classification/` to `sys.path` so the helpers import cleanly when the notebook is run from `notebooks/`. We use `experiments.summarize` and `experiments.print_summary_table` to format the per-experiment tables consistently.

In [ ]:
import json, os, sys
from pathlib import Path

PART_C = os.path.abspath('../part_c_rnn_classification')
if PART_C not in sys.path:
    sys.path.insert(0, PART_C)

%matplotlib inline

from experiments import MODEL_CONFIGS, summarize, print_summary_table

RESULTS_DIR = Path(PART_C) / 'results'
MODEL_ORDER = [c['name'] for c in MODEL_CONFIGS]

def load(name):
    return json.loads((RESULTS_DIR / f'{name}.json').read_text())

print('available results:', sorted(p.name for p in RESULTS_DIR.glob('*.json') if not p.name.startswith('_')))

## C.1 — Baseline: 6 models × 3 seeds, AG News, MAX_WORDS=25

Trainable embeddings, no pre-training. Results table reports **mean accuracy ± std** (across 3 seeds), trainable parameter count, and **wall-clock seconds per epoch** (averaged across seeds).

In [ ]:
c1 = load('c1_baseline')
print(f'runs: {len(c1)}  (= 6 architectures × 3 seeds)')
print_summary_table(summarize(c1), MODEL_ORDER)

**Observations.**
- All six architectures land within a tiny **0.6-point band (88.0 – 88.6%)**. With short context (25 tokens) and trainable-from-scratch embeddings, architecture choice barely matters.
- Best: **1Bi-RNN (88.6%)**; worst: 1Bi-LSTM (88.0%). The differences are barely above seed noise (max std ≈ 0.006).
- LSTMs (`1LSTM` 1.53 s/ep, `1Bi-LSTM` 1.81 s/ep) are actually *faster* than equivalent vanilla RNNs (`1RNN` 1.77 s/ep, `1Bi-RNN` 2.18 s/ep). PyTorch's LSTM kernel is well-optimized; PyTorch's vanilla RNN kernel is less so.
- Parameter counts are dominated by the embedding (~1.96M for the 19,635-token vocab × 100d). The recurrent layer adds 10 k – 200 k.
- vs Part B: best Part B (SVM word 1-grams) hit **0.9196** on the same dataset. Plain RNNs *underperform* the SVM by ~3 points here.

## C.2 — CPU vs accelerator (MPS) timing

1RNN and 1LSTM trained on CPU and on Apple-MPS for 5 epochs each (single seed). We're measuring **sec/epoch only**; accuracy is irrelevant here. On a CUDA T4 (Colab) the speed-up would be larger still — MPS has higher per-kernel-launch overhead than CUDA.

In [ ]:
c2 = load('c2_cpu_vs_gpu')
by = {(r['name'], r['device']): r['sec_per_epoch'] for r in c2}
header = f"{'model':<10} | {'CPU s/ep':>10} | {'MPS s/ep':>10} | {'speed-up':>9}"
sep = '-' * len(header)
print(sep); print(header); print(sep)
for name in ['1RNN', '1LSTM']:
    cpu_t, acc_t = by[(name, 'cpu')], by[(name, 'mps')]
    print(f'{name:<10} | {cpu_t:>10.2f} | {acc_t:>10.2f} | {cpu_t/acc_t:>8.1f}x')
print(sep)

**Observations.**
- **1RNN: 1.4× speed-up on MPS.** The per-step computation is small enough that MPS launch overhead eats most of the gain.
- **1LSTM: 3.8× speed-up on MPS.** LSTM has 4 gates → 4× more matmuls per step. The accelerator gain scales with arithmetic intensity.
- Practical takeaway: *the bigger the model, the bigger the GPU benefit.* Tiny networks barely benefit; deep multi-layer biLSTMs benefit a lot.

## C.3 — t-SNE of the embeddings learned by one trained 1RNN

Same 28 words as A.6, projected via t-SNE (perplexity=5, init='pca', random_state=42). One A.6 word didn't pass the `min_freq=10` cutoff in our AG News vocab — the script reports which one. Compare visually with `notebooks/part_a_embeddings.ipynb` A.6.

In [ ]:
from IPython.display import Image
Image(filename='../part_c_rnn_classification/figures/tsne_rnn_learned_c3.png')

**Observations vs A.6 (GloVe).**
- The RNN-learned plot is **much fuzzier**. GloVe was trained on **billions** of tokens with a co-occurrence objective → general semantic similarity. Our 1RNN was trained on 120k news docs with a **classification** objective → embeddings only need to be useful for separating the 4 AG News topics, *not* for generic semantics.
- For words rare in news (e.g. `lesson`, `homework`, `curriculum`), the embeddings stay close to their random init — t-SNE has no signal to recover.
- *Some* structure persists where it matters for topic discrimination: `stock`+`market`, `government`+`office`+`employee`+`manager`, `technology`+`learning`+`investment`.
- This contrast directly answers the assignment's question: **pre-trained embeddings = general semantic, task-trained embeddings = narrow task-aligned.** C.5 / C.6 (GloVe init) sit in the middle.

## C.4 — Repeat C.1 with MAX_WORDS = 50

Same models / same hyper-parameters except the input sequence length doubles (25 → 50). Tests how much each architecture benefits from longer context.

In [ ]:
c4 = load('c4_max_words_50')
print_summary_table(summarize(c4), MODEL_ORDER)

print('\n--- delta vs C.1 (mean accuracy) ---')
s1, s4 = summarize(c1), summarize(c4)
for n in MODEL_ORDER:
    d = s4[n]['mean_accuracy'] - s1[n]['mean_accuracy']
    print(f'  {n:<10} {s1[n]["mean_accuracy"]:.4f} -> {s4[n]["mean_accuracy"]:.4f}  ({d:+.4f})')

**Observations.**
- **Vanilla 1RNN gets *worse*** (-1.24 pts) — textbook **vanishing-gradient** effect. The 1RNN std also jumps from 0.003 → 0.011 (≈4× variance), classic instability with longer back-prop chains.
- **Every gated/bidirectional variant improves**, by +0.9 to +2.1 pts. LSTM gates do exactly what they're advertised to do: keep gradient signal alive across longer sequences.
- **LSTMs now decisively beat RNNs** (gap ~3 points between best LSTM and worst RNN). At MAX_WORDS=25 they were tied; at 50 the gating earns its keep.
- Best result jumps from 88.6% (1Bi-RNN @ 25) to **90.2% (2Bi-LSTM @ 50)** — narrows the Part-B SVM gap to ~1.8 pts.
- Sec/epoch ~doubles, as expected.
- Param counts unchanged — vocab and hidden dim untouched.

## C.5 — Repeat C.1 with GloVe-6B-100d initialization (trainable)

Same C.1 setup; the embedding layer's initial weights come from `glove-wiki-gigaword-100`. Embeddings are **trainable** (still get updated during training). OOV tokens get small random init; `<PAD>` stays zero.

In [ ]:
c5 = load('c5_glove_init')
print_summary_table(summarize(c5), MODEL_ORDER)

print('\n--- delta vs C.1 (mean accuracy) ---')
s5 = summarize(c5)
for n in MODEL_ORDER:
    d = s5[n]['mean_accuracy'] - s1[n]['mean_accuracy']
    print(f'  {n:<10} {s1[n]["mean_accuracy"]:.4f} -> {s5[n]["mean_accuracy"]:.4f}  ({d:+.4f})')

**Observations.**
- **Every model gains ~1-2 points uniformly** — pretrained init is a "rising-tide" effect. Largest gain: **1Bi-LSTM (+2.18)**. Smallest: **1Bi-RNN (+0.92)**. Gating mechanisms exploit a good initial semantic structure more than vanilla bi-RNN does.
- **1RNN gains a lot (+1.36)**: the weakest base learner benefits most from a warm start.
- **All six architectures still cluster tightly (89.5 - 90.2%)** — about the same 0.6-point spread as C.1. Architecture-vs-init interaction is small; what matters is the existence of a good init.
- Variance generally drops (or stays flat) — a warm-started model has less run-to-run noise.
- Param count and sec/epoch are essentially unchanged from C.1 — embedding init only sets *initial values*, not the optimizer's structure.

## C.6 — Repeat C.5 with FROZEN GloVe embeddings

Same as C.5 but the embedding layer's weights are not updated during training (`requires_grad = False`). Only the recurrent cells and the linear head are trained. Trainable parameter count drops from ~2 M → 10 k - 185 k — a **10-200× reduction**.

In [ ]:
c6 = load('c6_glove_frozen')
print_summary_table(summarize(c6), MODEL_ORDER)

print('\n--- delta vs C.5 (mean accuracy) ---')
s6 = summarize(c6)
for n in MODEL_ORDER:
    d = s6[n]['mean_accuracy'] - s5[n]['mean_accuracy']
    print(f'  {n:<10} {s5[n]["mean_accuracy"]:.4f} -> {s6[n]["mean_accuracy"]:.4f}  ({d:+.4f})')

**Observations.**
- **Vanilla RNNs lose** (-0.6 to -1.4 pts), **LSTMs win** (+0.4 to +0.8 pts). Two complementary explanations:
  - **Regularization.** Trainable embeddings on a not-huge dataset can drift from their pretrained meaning, eroding the GloVe head-start. Freezing forces the model to use the GloVe semantics as-is.
  - **Capacity needs to be in the right place.** LSTMs' gates are expressive enough to extract task-relevant signal from *fixed* inputs. Vanilla RNN cells aren't — they need the embeddings to do part of the discrimination work.
- **2Bi-LSTM wins decisively** at **0.9086 ± 0.0003**, the **best Part C result**. Variance also collapses — frozen embeddings remove a major source of run-to-run noise.
- **Sec/epoch slightly faster** (5-15 %) — no gradient through 1.96 M frozen embedding params.
- The Part B SVM is still ahead (0.9196), but the gap is now only ~1 point.

## C.7 — Repeat C.1 on IMDB (binary sentiment, 80/20 split)

Same architectures and hyper-parameters as C.1, but on the IMDB 50k movie-review dataset (deterministic 80/20 split → 40 k train / 10 k test). 2-class (negative / positive) instead of 4-class. Trainable embeddings, no pre-training.

**Important caveat.** IMDB reviews are long (typically 200-500 words) but we cap at MAX_WORDS=25 per the assignment — we are **looking only at the first 25 tokens of each review**. This is an aggressive truncation.

In [ ]:
c7 = load('c7_imdb')
print_summary_table(summarize(c7), MODEL_ORDER)

**Observations.**
- **Accuracy ~70-72%** — far below AG News (~88-90%). Two reasons compound:
  1. With MAX_WORDS=25 we throw away ~90% of each review's content. Sentiment cues distribute through the whole review (negations, contrast, late twists); we see only the opening.
  2. Sentiment is genuinely harder than topic — "matchstick / coach / league" *guarantees* Sports; positive vs negative often hinges on a single late phrase (`...but ultimately disappointing`).
- **Surprisingly, vanilla 1RNN ties or beats every gated/bidirectional variant** (0.7162 vs 0.70-0.71 for the rest). With 25 tokens there isn't much sequence to model — every architecture is effectively doing **bag-of-words sentiment** on the first 25 tokens, and the simplest cell wins.
- **Bidirectional and 2-layer don't help** because there's no real long-range structure to capture in 25 tokens. 
- **Vocab is much bigger than AG News** at the same `min_freq=10` (~25.5 k vs 19.6 k). Movie reviews use richer language than news headlines → param counts jump to ~2.5-2.7 M.
- *To really test sequence modeling on IMDB,* you'd need MAX_WORDS=200+. The C.1 / C.7 design exposes how MAX_WORDS interacts with the difficulty of the task.

## Cross-experiment summary — accuracy and parameters

Two side-by-side tables to compare across all experiments at once. The accuracy table uses **mean ± std** over 3 seeds; the parameters table reports **trainable** parameter count (which is why C.6 shows much smaller values — the 1.96 M frozen embedding is not counted).

In [ ]:
# All experiments to compare (skipping C.2 since it only trains 2 of the 6 models)
EXPERIMENTS = [
    ('C.1', 'c1_baseline'),
    ('C.4', 'c4_max_words_50'),
    ('C.5', 'c5_glove_init'),
    ('C.6', 'c6_glove_frozen'),
    ('C.7', 'c7_imdb'),
]
loaded = {tag: summarize(load(name)) for tag, name in EXPERIMENTS}

def print_grid(metric_label, fmt, value_fn):
    print(f'\n=== {metric_label} ===')
    header = f"{'model':<10} | " + ' | '.join(f'{tag:>14}' for tag, _ in EXPERIMENTS)
    print(header); print('-' * len(header))
    for n in MODEL_ORDER:
        row = f'{n:<10} | '
        row += ' | '.join(f'{value_fn(loaded[tag][n]):>14{fmt}}' for tag, _ in EXPERIMENTS)
        print(row)

print_grid('Mean Accuracy ± Std', '', lambda r: f"{r['mean_accuracy']:.4f}±{r['std_accuracy']:.4f}")
print_grid('Trainable Parameters', ',', lambda r: r['n_params'])
print_grid('Sec / epoch (avg)', '.2f', lambda r: r['sec_per_epoch'])

**Final cross-experiment readings.**

1. **Architecture matters less than hyper-parameter choices on AG News.** Across C.1 / C.4 / C.5 / C.6 the 6 architectures consistently cluster within a 1-2 point band. The big movers are MAX_WORDS (sequence length) and embedding strategy (random / GloVe-trainable / GloVe-frozen) — not the cell type.

2. **LSTM > vanilla RNN, but only when there's enough sequence to model.** At MAX_WORDS=25 they tie; at MAX_WORDS=50 LSTMs pull ahead by 1.5-3 points; on IMDB-25 they tie again because there's no real sequence anyway.

3. **GloVe init helps every model uniformly (+1-2 pts).** Freezing it helps LSTMs further but hurts vanilla RNNs — counter-intuitive but reproducible across seeds.

4. **Best Part C result on AG News: 2Bi-LSTM with frozen GloVe (90.86%).** Still ~1 point behind the Part B SVM word-1-grams baseline (91.96%). For news topic classification the lexical signal is so strong that classical baselines remain competitive.

5. **C.7 is the cautionary tale.** RNNs can only beat baselines when the *task* is dominated by sequential structure *and* you give the model enough context to see it. Truncating IMDB to 25 tokens kills both.